In [ ]:
# Fama-French 三因子模型实证分析（IBM 股票）
#下载 IBM 数据 → 计算收益率 → 合并三因子数据 → OLS 回归 → 三因子模型检验
import os
os.chdir('Desktop')
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
ibm=yf.download('IBM',start='2010-01-01',end='2020-12-31',multi_level_index=False)
#ibm=yf.download('IBM',start='2010-01-01',end='2020-12-31')
#print(ibm.head())
#multi_level_index=False = 单层列名 = 方便直接用 ibm ['Close']
#计算ibm的返回值
ibm['ret']=ibm['Close'].pct_change()#简单日收益率
print(ibm.head())
infile='http://datayyy.com/data_pickle/ff3daily.pkl'
ff3=pd.read_pickle(infile) #read_pickle() 读取二进制表格数据
print(ff3.head())
df=ibm.merge(ff3,left_on=ibm.index,right_on=ff3.index)
print(df.head())
#drop all null observation
df=df.dropna()
print(df.columns)
pd.set_option('display.max_columns',None) #显示所有列，不要把中间的列隐藏成省略号
y = df['ret'] - df.RF #df.RF：无风险利率，y是超额收益
x = df[["MKT_RF", "SMB", "HML"]]
#市场超额收益（市场收益率 - 无风险利率）；市值因子（小盘股 - 大盘股）；价值因子（高账面市值比 - 低账面市值比）
results=sm.add_constant(x)
results=sm.OLS(y, x).fit()
print(results.summary())


In [ ]:
#F分布可视化 + CDF与PPF互逆运算
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
x = np.arange(0, 4, 0.01) # 1. 生成x轴数值：从0到4，步长0.01
# 2. F分布的两个自由度（固定）
df1 = 30   # 分子自由度 dfn
df2 = 20   # 分母自由度 dfd
y = stats.f.pdf(x, df1, df2) # 3. 计算 F分布 的 概率密度函数 PDF（真正的密度分布）
# 4. 绘制F分布密度图
plt.title('F-distribution PDF')               # 标题
plt.xlabel('x values')                        # x轴标签
plt.ylabel('F-density (PDF)')                 # y轴标签（修正为正确名称）
plt.plot(x, y)                                # 画图
plt.show()                                    # 显示图片
# CDF：给定x值，求 累积概率 P(F ≤ x)
x_input = 1
prob = stats.f.cdf(x_input, df1, df2)
print("x = 1 时的累积概率 =", prob)
# PPF：给定累积概率，求 对应的x值（CDF的逆运算）
x_output = stats.f.ppf(prob, df1, df2)
print("根据概率反查x =", x_output)  # 结果会回到 1.0，验证互逆关系



In [ ]:
#按年度计算 IBM 股票的 Beta 系数
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
# 1. 下载 IBM 股票数据（单层列名，方便使用）
ibm = yf.download('IBM', start='2010-01-01', end='2020-12-31', multi_level_index=False)
ibm['ret'] = ibm['Close'].pct_change() # 2. 计算 IBM 日收益率
ibm['year'] = ibm.index.year # 3. 从日期索引中提取年份（用于按年分组回归）
# 4. 下载标普500指数数据（代表市场组合）
sp500 = yf.download('^GSPC', start='2010-01-01', end='2020-12-31', multi_level_index=False)
sp500['mkRet'] = sp500['Close'].pct_change()# 5. 计算市场收益率（标普500收益率）
# 6. 按日期合并股票数据与市场数据
df = ibm.merge(sp500, left_on=ibm.index, right_on=sp500.index)
df = df.dropna()
years = np.unique(df.year)# 8. 获取所有不重复的年份
betas = [] # 9. 创建空列表，用于存储每年的 Beta 系数
# 10. 【核心】按年循环，每年做一次CAPM回归，计算年度 Beta
for year in years:
    df2 = df[df.year == year] # 筛选当年的数据
    y = df2['ret'] # 被解释变量：股票收益率
    x = df2['mkRet'] # 解释变量：市场收益率
    x = sm.add_constant(x)
    results = sm.OLS(y, x).fit()
    beta = round(results.params[1], 3)     # 提取斜率系数，即 Beta（市场风险系数）
    betas.append(beta)
# 11. 把年份和对应Beta组合成DataFrame
annualbetas = pd.DataFrame([years, betas]).T
annualbetas.columns = ['years', 'beta']
print(annualbetas)


In [ ]:
# 夏普比率（Sharpe Ratio）计算代码，计算 IBM 股票全周期、近5年、年度化夏普比率
import pandas as pd
import numpy as np
# 1. 全样本夏普比率计算
df3 = ibm.merge(ff3, left_on=ibm.index, right_on=ff3.index, how='inner')
# 计算收益率标准差（风险）、无风险利率均值与标准差
meanret = round(df3.ret.mean(), 5)    # 股票收益率标准差（原代码误用std，正确应为mean）
std = round(df3.RF.std(), 5)         # 无风险利率标准差
meanrf = round(df3.RF.mean(), 5)     # 无风险利率均值
# 夏普比率公式：(股票平均收益率 - 无风险利率均值) / 收益率标准差
sharpe = round((meanret - meanrf) / std, 5)
print(f'ibm 全样本夏普比率 = {sharpe}')

# 2. 近5年（1260个交易日）夏普比率，取最近5年数据（美股年均252交易日，5年≈1260天）
df2 = ibm.tail(1260)
# 合并无风险利率数据
df3 = df2.merge(ff3, left_on=df2.index, right_on=ff3.index, how='inner')
meanret = round(df3.ret.mean(), 5)
std = round(df3.RF.std(), 5)
meanrf = round(df3.RF.mean(), 5)
sharpe = round((meanret - meanrf) / std, 5)
print(f'ibm 近5年夏普比率 = {sharpe}')

# 3. 年度化夏普比率计算，计算对数收益率（更适合年度复利计算）
ibm['logret'] = np.log(ibm['Close'].pct_change() + 1)
# 按年度分组求和 → 转换为年度简单收益率
annualret = np.exp(ibm['logret'].groupby(ibm['year']).sum()) - 1
annualret2 = pd.DataFrame(annualret)
annualret2.columns = ['annualret']  # 重命名列
print('IBM 年度收益率：')
print(annualret2)
# 读取年度 Fama-French 无风险利率数据
infile = 'http://datayyy.com/data_pickle/ff3annual.pkl'
ff3_2 = pd.read_pickle(infile)
# 合并年度收益率与年度无风险利率
df4 = annualret2.merge(ff3_2, left_index=True, right_index=True, how='inner')
# 计算年度化夏普比率
meanret = round(df4.annualret.mean(), 5)
std = round(df4.RF.std(), 5)
meanrf = round(df4.RF.mean(), 5)
sharpe = round((meanret - meanrf) / std, 5)
print(f'ibm 年度化夏普比率 = {sharpe}')

In [ ]:
#特雷诺比率 = (平均年化收益率 - 平均无风险利率) / Beta，衡量单位系统性风险带来的超额收益
# 1. 计算IBM股票的Beta系数（基于年度收益率）
# 将资产收益率和市场超额收益率组合成矩阵
x = np.stack((df4.annualret, df4.MKT_RF), axis=0) #把多个数组 / 序列，沿着新的维度堆叠在一起
cov = np.cov(x) # 计算协方差矩阵
# Beta = 资产与市场收益协方差 / 市场收益方差
beta = cov[0, 1] / cov[1, 1]
beta = round(beta, 5)
print(f'IBM 年度Beta系数 = {beta}')
# 2. 计算平均收益率与无风险利率
meanret = round(df4.annualret.mean(), 5)    # IBM年化平均收益率
meanrf = round(df4.RF.mean(), 5)            # 年度无风险利率均值
# 3. 计算特雷诺比率
treynor = round((meanret - meanrf) / beta, 5)
print(f'IBM 年度特雷诺比率 = {treynor}')

In [ ]:
#LPSD（Lower Partial Standard Deviation）下偏标准差是只计算收益率低于无风险利率（或目标收益）时的波动程度，专门用来衡量资产下行风险的指标
import pandas as pd
import numpy as np
np.random.seed(12345)    # 固定随机种子，结果可复现
mean = 0.05             # 平均收益
rf = 0.02                 # 无风险利率
std = 0.02                # 标准差
n = 100                   # 生成100个随机收益
# 生成正态分布的随机收益数据
x = np.random.normal(loc=mean, scale=std, size=n)
print("随机收益序列：\n", x)
# 筛选出 收益 - 无风险利率 < 0 的样本（下行收益）
y = x[x - rf < 0]
print("\n低于无风险利率的下行收益：\n", y)

# 手工计算 LPSD
total = 0.0
for r in y:
    total += (r - rf) ** 2  # 下行偏差平方和
LPSD = round(np.sqrt(total / 63), 5)  # 除以63（季度交易日）
print("\n演示 LPSD =", LPSD)

# 2. 封装 LPSD 计算函数（通用版）
def lpsd_f(returns, rm):
    """
    计算下偏标准差 LPSD
    returns: 收益率序列
    rm: 无风险利率/基准收益
    """
    y = returns[returns < rm]
    total = 0.0
    m = len(y)
    
    # 计算下行偏差平方和
    for r in y:
        total += (r - rm) ** 2  
    # 无偏方差：除以 m-1；如果 m<2 会报错，这里默认数据足够
    var = total / (m - 1)
    lpsd = round(np.sqrt(var), 5)  # 开平方 → LPSD
    return lpsd


In [ ]:
# 3. 真实数据：IBM 月度 LPSD 计算
annual_risk_free = 0.01  # 年化无风险利率
rf_monthly = annual_risk_free / 12  # 转为月度无风险利率
path = 'http://datayyy.com/data_pickle/'
df5 = pd.read_pickle(path + 'ibmmonthly.pkl')
# 计算月度收益率
df5['ret'] = df5['adj close'].pct_change()
# 计算 LPSD（剔除缺失值）
lpsd_result = lpsd_f(df5['ret'].dropna(), rf_monthly)
print(f"\nIBM 月度 LPSD = {lpsd_result}")